# Non-Linear Economy Estimation (ANN)

Please note that this notebook uses a venv which points to a base python version of **3.13**, some functionality may be limited if using an older version of python.

## All Imports

- Vanilla Modules:
    - typing (optional)
        - just for static type checking and code hygeine, not functionally required.
    - dataclasses (optional)
        - class type of choice for SVAR return object, also not necessary but functionally optimal.
    - datetime
        - Used for datetime.now() to set the end date bound for out of sample data.
    
- 3rd party:
    - fedfred
        - This is the library which I wrote, published, and currently maintain under the MIT License
        - Makes API requests to the FRED database and returns strongly typed and structured objects as well as dataframes to reduce boilerplate code.
    - pandas
        - dataframe backend of choice, if computational speed or parallalelization becomes a necessity the fedfred backend accomodates dask and polars.
    - numpy
        - for mathematical ops and pandas interop
    - statsmodels
        - for OLS and potentially VAR model if necessary.
    - matplotlib
        - Used for plotting results.
    - 

In [1]:
%pip install scikit-learn seaborn torch torchvision gymnasium stable-baselines3 --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
import fedfred as fed
import pandas as pd
import numpy as np
import statsmodels.api as sm
import torch
import matplotlib.pyplot as plt

In [3]:
%matplotlib widget

## Data From Source Package

In [4]:
%pip install -e ../ --quiet

Note: you may need to restart the kernel to use updated packages.


In [5]:
import autonomous_fed as afed

In [8]:
le_solver = afed.LinearEnvironmentSolver(fred_key="7ab121fb17773e187bb6508e83e411da")

# Data Check
print (le_solver.historical_data.head())
print (le_solver.historical_data.tail())

             pi         y     i
date                           
1987Q3  2.66122 -0.388640  6.84
1987Q4  2.91876  0.526730  6.92
1988Q1  3.06577  0.260731  6.66
1988Q2  3.35274  0.788853  7.16
1988Q3  3.80207  0.595530  7.98
             pi         y     i
date                           
2006Q2  3.35642  1.319357  4.91
2006Q3  3.13805  0.978513  5.25
2006Q4  2.66316  1.380249  5.25
2007Q1  2.91019  1.210046  5.26
2007Q2  2.72883  1.336951  5.25


## Model Buildout

In [7]:
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

class SimpleNeuralNetwork(torch.nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(SimpleNeuralNetwork, self).__init__()
        self.fc1 = torch.nn.Linear(input_size, hidden_size)
        self.relu = torch.nn.ReLU()
        self.fc2 = torch.nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

# Prepare your data using the variables from linear_environment.ipynb
# Create feature matrix (X) and target variable (y)
X = le_solver.historical_data[['pi', 'y', 'i']].dropna()  # Features: inflation, output gap, interest rate
y = X['i'].shift(-1).dropna()  # Target: next period's interest rate
X = X[:-1]  # Remove last row to match y length

# Split into train/test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale the data
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled = scaler_X.transform(X_test)
y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1)).flatten()
y_test_scaled = scaler_y.transform(y_test.values.reshape(-1, 1)).flatten()

# Convert to PyTorch tensors
X_train = torch.FloatTensor(X_train_scaled)
X_test = torch.FloatTensor(X_test_scaled)
y_train = torch.FloatTensor(y_train_scaled)
y_test = torch.FloatTensor(y_test_scaled)

# Initialize model
input_dim = X_train.shape[1]  # Number of features (3)
hidden_dim = 20
output_dim = 1  # Predicting one value (interest rate)

model = SimpleNeuralNetwork(input_dim, hidden_dim, output_dim)
criterion = torch.nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)  # Adam often works better than SGD

# Train the model
num_epochs = 1000

for epoch in range(num_epochs):
    model.train()
    optimizer.zero_grad()
    
    # Forward pass
    outputs = model(X_train).squeeze()  # Remove extra dimension
    loss = criterion(outputs, y_train)
    
    # Backward pass and optimize
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 100 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

# Evaluate the model
model.eval()
with torch.no_grad():
    test_outputs = model(X_test).squeeze()
    test_loss = criterion(test_outputs, y_test)
    print(f'Test Loss: {test_loss.item():.4f}')
    
    # Convert back to original scale for interpretation
    test_predictions = scaler_y.inverse_transform(test_outputs.numpy().reshape(-1, 1))
    actual_values = scaler_y.inverse_transform(y_test.numpy().reshape(-1, 1))
    
    print(f'Sample predictions vs actual:')
    for i in range(min(5, len(test_predictions))):
        print(f'Predicted: {test_predictions[i][0]:.4f}, Actual: {actual_values[i][0]:.4f}')

Epoch [100/1000], Loss: 0.1628
Epoch [200/1000], Loss: 0.0764
Epoch [300/1000], Loss: 0.0415
Epoch [400/1000], Loss: 0.0330
Epoch [500/1000], Loss: 0.0311
Epoch [600/1000], Loss: 0.0304
Epoch [700/1000], Loss: 0.0299
Epoch [800/1000], Loss: 0.0295
Epoch [900/1000], Loss: 0.0291
Epoch [1000/1000], Loss: 0.0288
Test Loss: 0.0215
Sample predictions vs actual:
Predicted: 5.5498, Actual: 6.0200
Predicted: 6.3168, Actual: 6.9200
Predicted: 2.8250, Actual: 3.0000
Predicted: 5.6995, Actual: 5.8000
Predicted: 3.5145, Actual: 3.7700
